# World Bank data download code

In [ ]:
# (1) Paths, country-year table, and variables
import os
import time
from typing import Dict

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

PROJECT_ROOT = "/Users/karwailin/Desktop/OAwalk"
ISO_PATH = os.path.join(PROJECT_ROOT, "data", "iso36.csv")
LAST_SURVEY_YEAR_PATH = os.path.join(PROJECT_ROOT, "results", "model_prep", "last_survey_year_by_country.csv")
OUT_DIR = os.path.join(PROJECT_ROOT, "data", "wb")
OUT_PATH = os.path.join(OUT_DIR, "wb_last_survey_year.csv")
LE_OUT_PATH = os.path.join(PROJECT_ROOT, "data", "wb", "wb_life_expectancy_last_survey_year.csv")

INDICATORS: Dict[str, str] = {
    "SP.POP.TOTL": "total_population",
    "SP.POP.65UP.TO.ZS": "age65_share",
}

LE_INDICATORS: Dict[str, str] = {
    "SP.DYN.LE00.IN": "le",
    "SP.DYN.LE00.FE.IN": "female",
    "SP.DYN.LE00.MA.IN": "male",
}

os.makedirs(OUT_DIR, exist_ok=True)

iso = pd.read_csv(ISO_PATH).rename(columns={"isocountry_c": "country"})
last_year = pd.read_csv(LAST_SURVEY_YEAR_PATH)
iso["country"] = iso["country"].astype(str).str.strip()
iso["iso3c"] = iso["iso3c"].astype(str).str.strip().str.upper()
last_year["country"] = last_year["country"].astype(str).str.strip()
last_year["last_survey_year"] = pd.to_numeric(last_year["last_survey_year"], errors="coerce")

country_year = iso[["country", "iso3c", "dataset"]].merge(last_year, on="country", how="left")
missing_year = country_year.loc[country_year["last_survey_year"].isna(), "country"].tolist()
if missing_year:
    raise ValueError(f"Countries in iso36 without last_survey_year: {missing_year}")
country_year["year"] = country_year["last_survey_year"].round().astype(int)

COUNTRY_CODES = sorted(country_year["iso3c"].dropna().unique())
START_YEAR = int(country_year["year"].min())
END_YEAR = int(country_year["year"].max())

print("Countries:", len(COUNTRY_CODES))
print("Years needed:", f"{START_YEAR}-{END_YEAR}")
print("OUT_PATH:", OUT_PATH)
print("LE_OUT_PATH:", LE_OUT_PATH)
country_year.head()

Countries: 36
Years needed: 2011-2024
OUT_PATH: /Users/karwailin/Desktop/OAwalk/data/wb/wb_last_survey_year.csv
LE_OUT_PATH: /Users/karwailin/Desktop/OAwalk/data/wb/wb_life_expectancy_last_survey_year.csv


,country,iso3c,dataset,last_survey_year,year
0,Australia,AUS,ALSA,2014.0,2014
1,Austria,AUT,SHARE,2022.0,2022
2,Belgium,BEL,SHARE,2022.0,2022
3,Bulgaria,BGR,SHARE,2022.0,2022
4,China,CHN,CHARLS,2018.0,2018


In [7]:
# (2) World Bank API helpers
WB_BASE = "https://api.worldbank.org/v2"

session = requests.Session()
retry = Retry(
    total=5,
    connect=5,
    read=5,
    backoff_factor=1.5,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
)
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))


def fetch_indicator_for_countries(code: str, countries, start_year: int, end_year: int, sleep: float = 0.2) -> pd.DataFrame:
    country_expr = ";".join(countries)
    all_rows = []
    page = 1

    while True:
        url = f"{WB_BASE}/country/{country_expr}/indicator/{code}"
        params = {
            "format": "json",
            "per_page": 20000,
            "date": f"{start_year}:{end_year}",
            "page": page,
        }
        resp = session.get(url, params=params, timeout=(10, 180))
        resp.raise_for_status()
        data = resp.json()
        if not isinstance(data, list) or len(data) < 2:
            break

        meta, rows = data[0], data[1]
        if not rows:
            break

        all_rows.extend(rows)
        if page >= int(meta.get("pages", 1)):
            break
        page += 1
        time.sleep(sleep)

    df = pd.DataFrame(all_rows)
    if df.empty:
        return df

    out = df[["countryiso3code", "country", "date", "value"]].copy()
    out["country"] = out["country"].apply(lambda x: x.get("value") if isinstance(x, dict) else x)
    out["date"] = pd.to_numeric(out["date"], errors="coerce").astype("Int64")
    out = out.rename(columns={"countryiso3code": "iso3c", "value": code})
    out["iso3c"] = out["iso3c"].astype(str).str.upper()
    return out


def download_wb_wide(indicators: Dict[str, str], out_path: str) -> pd.DataFrame:
    merged = None
    for code in indicators.keys():
        print("Fetching:", code)
        df = fetch_indicator_for_countries(code, COUNTRY_CODES, START_YEAR, END_YEAR)
        if df.empty:
            print("No data for", code)
            continue
        if merged is None:
            merged = df
        else:
            merged = pd.merge(merged, df, on=["iso3c", "country", "date"], how="outer")

    if merged is None:
        raise RuntimeError("No World Bank data downloaded")

    for code, new_name in indicators.items():
        if code in merged.columns:
            merged = merged.rename(columns={code: new_name})

    exact = country_year.merge(merged, left_on=["iso3c", "year"], right_on=["iso3c", "date"], how="left")
    exact = exact.drop(columns=["date"])
    if "country_y" in exact.columns:
        exact = exact.rename(columns={"country_x": "country", "country_y": "wb_country"})

    value_cols = list(indicators.values())
    exact = exact[["country", "iso3c", "dataset", "last_survey_year", "year", "wb_country"] + value_cols]
    exact = exact.sort_values("country").reset_index(drop=True)
    exact.to_csv(out_path, index=False)
    print("Saved:", out_path)
    print("Shape:", exact.shape)
    return exact

In [8]:
# (3) Download required variables at each country's last survey year
wb_year = download_wb_wide(INDICATORS, OUT_PATH)
le_year = download_wb_wide(LE_INDICATORS, LE_OUT_PATH)

print("WB missing counts:")
print(wb_year[list(INDICATORS.values())].isna().sum())
print("Life expectancy missing counts:")
print(le_year[list(LE_INDICATORS.values())].isna().sum())

wb_year.head()

Fetching: SP.POP.TOTL
Fetching: SP.POP.65UP.TO.ZS
Fetching: SP.URB.TOTL.IN.ZS
Fetching: SP.URB.TOTL
Fetching: SP.POP.DPND.OL
Saved: /Users/karwailin/Desktop/OAwalk/data/wb/wb_last_survey_year.csv
Shape: (36, 11)
Fetching: SP.DYN.LE00.IN
Fetching: SP.DYN.LE00.FE.IN
Fetching: SP.DYN.LE00.MA.IN
Saved: /Users/karwailin/Desktop/OAwalk/data/wb/wb_life_expectancy_last_survey_year.csv
Shape: (36, 9)
WB missing counts:
total_population          0
age65_share               0
urban_population_share    0
urban_population          0
old_dep_ratio             0
dtype: int64
Life expectancy missing counts:
le        0
female    0
male      0
dtype: int64


,country,iso3c,dataset,last_survey_year,year,wb_country,total_population,age65_share,urban_population_share,urban_population,old_dep_ratio
0,Australia,AUS,ALSA,2014.0,2014,Australia,23475686,14.634393,86.722129,20358615,22.023452
1,Austria,AUT,SHARE,2022.0,2022,Austria,9041851,19.789751,68.998085,6238704,30.046347
2,Belgium,BEL,SHARE,2022.0,2022,Belgium,11680210,19.767193,87.489200,10218922,31.030182
3,Bulgaria,BGR,SHARE,2022.0,2022,Bulgaria,6465097,21.697297,73.479916,4750548,34.027780
4,China,CHN,CHARLS,2018.0,2018,China,1402760000,11.516209,61.500198,862700183,16.415305
